# BIOSTAT 707: Checkpoint 1
**Author:** Ahmed Hussain
**NetID:** ah418

# Packages and data used in this notebook:

Silva et al. 2012, Computing in Cardiology 39:245–248; Goldberger et al. 2000, Circulation 101(23):e215–e220; ODC-BY 1.0

In [ ]:
from pathlib import Path
import subprocess

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd()
assert (ROOT / "checkpoint1.ipynb").exists(), f"Run from the repo root, not {ROOT}"
DATA = ROOT / "data"
OUT = ROOT / "output"
OUT.mkdir(exist_ok=True)
print("ROOT:", ROOT)

## Check data

In [ ]:
def check_set_a(data_dir: Path) -> None:
    set_a = data_dir / "set-a"
    assert set_a.is_dir(), f"missing folder: {set_a}"
    records = sorted(set_a.glob("*.txt"))
    assert len(records) == 4000, f"expected 4000 record files in {set_a}, found {len(records)}"
    outcomes = data_dir / "Outcomes-a.txt"
    assert outcomes.is_file(), f"missing file: {outcomes}"
    print(f"set-a OK: {len(records)} record files; Outcomes-a.txt present")

check_set_a(DATA)

## Load

Each record file is a 3-column CSV (`Time, Parameter, Value`). The first six rows at `00:00` are admission descriptors (RecordID, Age, Gender, Height, ICUType, Weight); everything after is a time-stamped measurement. `load_set_a()` stacks all 4000 files, splits the descriptors into a one-row-per-record `static` table, and keeps the rest as `long` with `Time` converted to numeric hours since ICU admission.

In [ ]:
DESCRIPTORS = ["Age", "Gender", "Height", "ICUType", "Weight"]

def load_set_a(data_dir):
    files = sorted(data_dir.glob("set-a/*.txt"))
    df = pd.concat([pd.read_csv(f).assign(RecordID=int(f.stem)) for f in files], ignore_index=True)
    df["Hours"] = pd.to_timedelta(df["Time"] + ":00").dt.total_seconds() / 3600
    is_static = (df["Hours"] == 0) & df["Parameter"].isin(DESCRIPTORS + ["RecordID"])
    static = df[is_static].pivot(index="RecordID", columns="Parameter", values="Value")[DESCRIPTORS]
    return df[~is_static], static

long, static = load_set_a(DATA)
long.to_csv(OUT / "set-a_long.csv", index=False)
print(long.shape, static.shape)
long.head()

# High-level look at the parameters

In [ ]:
# number of unique parameters
print("Number of unique parameters:", long["Parameter"].nunique())

# list of unique parameters
print("List of unique parameters:", long["Parameter"].unique())

# Examining each parameter

We can look at each parameter to see how many patients have tha parameter measured, how many total obvservations, how many of the total patients have at least one observation, the median number of observations per patient, and the mean number of observations per patient (among patients who have at least one observation for that parameter).

In [ ]:
patients_per_parameter = (
    long.groupby("Parameter")["RecordID"]
    .nunique()
    .sort_values()
)

patient_coverage = (
    long.groupby("Parameter")["RecordID"]
    .nunique()
    .div(long["RecordID"].nunique())
    .mul(100)
    .sort_values()
)

# Median number of observations per patient,
# among patients who have at least one observation for that parameter
median_observations_per_patient = (
    long.groupby(["Parameter", "RecordID"])
    .size()
    .groupby("Parameter")
    .median()
)

# Mean number of observations per patient, among patients who have at least one observation for that parameter
# Round to 2 decimal places for better readability
mean_observations_per_patient = (
    long.groupby(["Parameter", "RecordID"])
    .size()
    .groupby("Parameter")
    .mean()
    .round(2)
)

# Merge patient coverage with observation counts and median observations per patient
coverage_df = pd.DataFrame({
    "Parameter": patients_per_parameter.index,
    "Patients": patients_per_parameter.values,
    "Coverage (%)": patient_coverage.values,
    "Observations": long.groupby("Parameter")["Value"].count().reindex(
        patients_per_parameter.index
    ).values,
    "Median observations/patient": median_observations_per_patient.reindex(
        patients_per_parameter.index
    ).values,
    "Mean observations/patient": mean_observations_per_patient.reindex(
        patients_per_parameter.index
    ).values
}).sort_values("Coverage (%)", ascending=False)

coverage_df

# Plotting coverage

Roughly speaking, there are about 7 sets of parameters that have similar coverage. We can plot the coverage of each parameter to see how many patients have that parameter measured.

In [ ]:
plt.figure(figsize=(12, 7))

patient_coverage.sort_values().plot(kind="barh")

plt.xlabel("Patients with at least one measurement (%)")
plt.ylabel("Parameter")
plt.title("Patient coverage by parameter")
plt.tight_layout()
plt.show()

# Basic statistics

In [ ]:
summary = (
    long.groupby("Parameter")["Value"]
    .agg(["count", "min", "max", "mean", "median"])
    .sort_index()
)

summary

# Inspecting individual parameters

Looking at the above table, we can see that some parameters have suspiciously low or high values. Below, we can inspect some parameters individually to confirm. Heart rate is an interesting parameter because aside from extreme cases, the value rarely drops below 40 or rises above 200. Even less likely, heart rate dropping to 0. Based on the above table, this happened many times. We could consider this as signal drop off or a patient being in cardiac arrest. Because I thought that the presence of 0 values could be meaningful to prediction of mortality, I did not remove them.

In [ ]:
long[long["Parameter"] == "HR"].nsmallest(10, "Value")


# Example of extremes for HR

We can see several times when HR dropped to 0 then came back up. There are also some instances where HR dropped to 0 and stayed there.

In [ ]:
# plot all HR values over time for all patients
plt.figure(figsize=(12, 6))
sns.lineplot(data=long[long["Parameter"] == "HR"], x="Hours", y="Value", hue="RecordID", legend=False)
plt.title("Heart Rate (HR) over Time for All Patients")
plt.xlabel("Hours")
plt.ylabel("Heart Rate (HR)")

# A note regarding AI and catching extreme values

I used Claude Code to examine the data and report the ranges and extremes of each parameter. The initial goal was to fine measurements that make no sense under any circumstances.

After looking through the list of parameters, what made the the most sense was to make rule set that is specific to each parameter. 

For the sake of succinctness, I show a couple of examples. We can look at the height and see that all data points see fine until we reach values over 205.7. Those extreme values are likely conversion errors between inches and centimeters. If we divide those values in that set by 2.54 (2.54 cm : 1 inch), we get much more reasonable values. Therefore, I decided that it would unreasonable to remove those values from the dataset given that they are relatively few, can be corrected, and are likely to be simple conversion errors.


Likewise for pH, we can see that there are values between 94 to 187 (which is impossible and unlikely to be simple entry errors) but there are also values of 733 to 735 (which are also impossble but very likely entry mistakes that require simply dividing by 100). 


In [ ]:
# table of nlargest height values in the static data
print(static.nlargest(10, "Height"))
print(long[long["Parameter"] == "pH"].nlargest(10, "Value"))

# Age distribution

In [ ]:
# plot age range distribution
plt.figure(figsize=(8, 6))
sns.histplot(static["Age"], bins=20, kde=True)
plt.title("Age Distribution of Patients")

# Boxplot of every parameter to see the distribution of values

In [ ]:
plt.figure(figsize=(14, 8))

sns.boxplot(
    data=long,
    x="Parameter",
    y="Value"
)

plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

# Rule set for each parameter

In [ ]:
LIMITS = {  # inclusive physically-possible range; any parameter not listed is a lab value and only needs to be > 0
    "SysABP": (1, 300), "DiasABP": (1, 300), "MAP": (1, 300),          # 0 = line disconnected; 300 = transducer ceiling
    "NISysABP": (1, 300), "NIDiasABP": (1, 300), "NIMAP": (1, 300),    # 0 = cuff failed to read
    "HR": (0, 300), "RespRate": (0, 80), "Urine": (0, np.inf),          # 0 = asystole (see rule below) / apnea / anuria
    "Temp": (13, 47), "pH": (6.5, 8), "PaCO2": (5, 250), "PaO2": (10, 700), "FiO2": (0.21, 1),
    "SaO2": (0, 100), "HCT": (0, 100), "GCS": (3, 15), "MechVent": (1, 1), "Albumin": (0, 7), "Weight": (5, 500),
    "Age": (15, 120), "Gender": (0, 1), "ICUType": (1, 4), "Height": (50, 275),   # descriptors; -1 = unknown
}

def clean(rows):
    p, v = rows["Parameter"], rows["Value"].copy()
    v = v.mask((p == "pH") & (v > 100), v / 100)            # 733 -> 7.33: decimal point lost
    v = v.mask((p == "Height") & (v > 275), v / 2.54)       # 431.8 -> 170: cm typed into an inches field
    v = v.mask((p == "Height") & (v < 3), v * 100)          # 1.8 -> 180: metres instead of cm
    repaired = v != rows["Value"]

    lo = p.map(lambda x: LIMITS.get(x, (0, np.inf))[0])
    hi = p.map(lambda x: LIMITS.get(x, (0, np.inf))[1])
    out_of_range = (v < lo) | (v > hi) | (~p.isin(LIMITS) & (v == 0))
    v = v.mask(out_of_range)

    last_beat = rows.loc[(p == "HR") & (v > 0)].groupby("RecordID")["Hours"].max()
    hr_artefact = (p == "HR") & (v == 0) & (rows["Hours"] < rows["RecordID"].map(last_beat))   # HR 0 then beats again = lead off

    bp = rows.assign(Value=v)[p.str.endswith("ABP")].pivot_table(index=["RecordID", "Time"], columns="Parameter", values="Value")
    key = pd.MultiIndex.from_frame(rows[["RecordID", "Time"]])
    dia_ge_sys = pd.Series(False, index=rows.index)
    for s, d in [("SysABP", "DiasABP"), ("NISysABP", "NIDiasABP")]:
        dia_ge_sys |= key.isin(bp.index[bp[d] >= bp[s]]) & p.isin([s, d])
    v = v.mask(hr_artefact | dia_ge_sys)

    reason = pd.Series("", index=rows.index)
    reason[repaired] = "unit/decimal repaired"
    reason[out_of_range] = "outside possible range"
    reason[hr_artefact] = "HR 0 with later heartbeat"
    reason[dia_ge_sys] = "diastolic >= systolic"
    return rows.assign(Value=v), rows.assign(NewValue=v, Reason=reason)[reason != ""]

# descriptors follow the same rules: put them in long format, clean everything once, split again
static_rows = static.stack().rename("Value").reset_index().assign(Time="00:00", Hours=0.0)
clean_all, changed = clean(pd.concat([long, static_rows], ignore_index=True))
is_static = (clean_all["Hours"] == 0) & clean_all["Parameter"].isin(DESCRIPTORS)
long_clean = clean_all[~is_static]
static_clean = clean_all[is_static].pivot(index="RecordID", columns="Parameter", values="Value")[DESCRIPTORS]

# changed.to_csv(OUT / "set-a_changed.csv", index=False)
# long_clean.to_csv(OUT / "set-a_long_clean.csv", index=False)
changed.groupby(["Parameter", "Reason"]).size().rename("n")

In [ ]:
n = long["RecordID"].nunique()
valid = long_clean.dropna(subset=["Value"])
missing = pd.DataFrame({
    "values": long.groupby("Parameter").size(),
    "values_set_NaN": long_clean["Value"].isna().groupby(long_clean["Parameter"]).sum(),
    "patients_missing": n - long.groupby("Parameter")["RecordID"].nunique(),
    "patients_missing_clean": n - valid.groupby("Parameter")["RecordID"].nunique(),
}).sort_values("patients_missing")
missing

# When are variables measured?

We can take a look at the heatmap below through it is difficult to see informative patterns. This is because the measures are compared on the same scale.

In [ ]:
hour = valid.assign(Hour=valid["Hours"].clip(upper=47.99).astype(int))
observed = hour.groupby(["Parameter", "Hour"])["RecordID"].nunique().unstack(fill_value=0) / n
observed = observed.loc[observed.mean(axis=1).sort_values(ascending=False).index]

plt.figure(figsize=(14, 10))
sns.heatmap(observed, cmap="viridis", cbar_kws={"label": "proportion of patients measured"})
plt.xlabel("Hour since ICU admission"); plt.ylabel("")
plt.title("When is each variable measured?")
plt.show()

# Normalized heatmap

We can a much clearer picture about the patterns of when each parameter is measured. Parameters such as Temp and GCS are measured on frequent and consistent intervals.

In [ ]:
hour = valid.assign(Hour=valid["Hours"].clip(upper=47.99).astype(int))
observed = hour.groupby(["Parameter", "Hour"])["RecordID"].nunique().unstack(fill_value=0)
observed = observed.loc[observed.sum(axis=1).sort_values(ascending=False).index]   # rows: most to least measured overall
relative = observed.div(observed.max(axis=1), axis=0)                               # each row scaled to its own busiest hour

plt.figure(figsize=(14, 10))
sns.heatmap(relative, cmap="viridis", cbar_kws={"label": "patients measured, relative to the variable's busiest hour"})
plt.xlabel("Hour since ICU admission"); plt.ylabel("")
plt.title("When is each variable measured? (each row relative to itself)")
plt.show()

# Wide table

In [ ]:
outcomes = pd.read_csv(DATA / "Outcomes-a.txt").set_index("RecordID")
STATS = ["count", "first", "last", "min", "max", "mean"]

summary = valid.groupby(["RecordID", "Parameter"])["Value"].agg(STATS).unstack("Parameter")
summary["count"] = summary["count"].fillna(0).astype(int)          # never measured = 0, not NaN
summary.columns = [p + "_" + s for s, p in summary.columns]         # (mean, HR) -> HR_mean
wide = static_clean.join(summary).join(outcomes)
wide.to_csv(OUT / "set-a_wide.csv")
print(wide.shape)
wide.head()

Proportion of patients with no valid measurement in 48 h

In [ ]:
import missingno as msno

measured = wide[DESCRIPTORS + list(wide.filter(like="_mean").columns)]
measured.columns = measured.columns.str.replace("_mean", "")

missing_pct = measured.isna().mean().mul(100).sort_values()
missing_pct.plot.barh(figsize=(8, 10), title="Patients with no valid measurement in 48 h (%)")
plt.show()

# Nullity Matrix

In [ ]:
measured = wide[DESCRIPTORS + list(wide.filter(like="_mean").columns)]
measured.columns = measured.columns.str.replace("_mean", "")

ax = msno.matrix(measured.sort_values("ICUType"), sparkline=False, figsize=(18, 9), fontsize=10)
ax.set_title("Nullity matrix: one row per patient (sorted by ICU type), white = no valid value in 48 h")
plt.show()

# Outcome summary

In [ ]:
print(outcomes["In-hospital_death"].value_counts().rename("patients"))
print("death rate:", outcomes["In-hospital_death"].mean().round(3))
display(outcomes.replace(-1, np.nan).describe().round(1))      # -1 = unknown (scores, LOS) or no death recorded (Survival)
wide.groupby("ICUType")["In-hospital_death"].agg(n="size", death_rate="mean").round(3)

554/4000 died (13.8 %). Death rate ranges from 4.9 % (cardiac surgery) to 18.6 % (medical ICU). Survival is known for only 1474 patients; SAPS-I/SOFA missing for ~150–190.

# Table 1

In [ ]:
from tableone import TableOne

t1 = wide.assign(
    Gender=wide["Gender"].map({0: "Female", 1: "Male"}),
    ICUType=wide["ICUType"].map({1: "Coronary", 2: "Cardiac surgery", 3: "Medical", 4: "Surgical"}),
    Ventilated=(wide["MechVent_count"] > 0).map({True: "Yes", False: "No"}),
    Outcome=wide["In-hospital_death"].map({0: "Survived", 1: "Died"}),
)
columns = ["Age", "Gender", "ICUType", "Height", "Weight", "Ventilated", "GCS_min", "HR_mean", "MAP_min",
           "Temp_max", "Creatinine_max", "Lactate_max", "Urine_count"]
TableOne(t1, columns=columns, categorical=["Gender", "ICUType", "Ventilated"], groupby="Outcome",
         nonnormal=["Creatinine_max", "Lactate_max", "Urine_count"], pval=True, missing=True)

Those who died were older (70 vs 63), more often ventilated (71 % vs 62 %), lower GCS_min (7.1 vs 8.4), higher creatinine and lactate. Height is missing for 1910, MAP_min for 1208 (no arterial line).

# Informative missingness: death rate if measured vs never measured

In [ ]:
death = wide["In-hospital_death"]
ever = pd.concat([wide[["Height"]].notna(), wide.filter(like="_count").gt(0).rename(columns=lambda c: c.replace("_count", ""))], axis=1)
informative = pd.DataFrame({
    "n_never": (~ever).sum(),
    "death_if_measured": ever.apply(lambda m: death[m].mean()),
    "death_if_never": ever.apply(lambda m: death[~m].mean()),
    # column showing absolute difference in death rate between measured and never measured
    "abs_diff": ever.apply(lambda m: abs(death[m].mean() - death[~m].mean()))
})
informative = informative[informative["n_never"] >= 30].sort_values("death_if_never")
informative.plot.barh(y=["death_if_measured", "death_if_never"], figsize=(8, 10), title="In-hospital death rate by whether the variable was ever measured")
plt.show()
informative.round(3)

Missingness can be informative bidirectionally.

Not having routine labs like WBC or BUN corresponds to a ~6 % death rate vs 14% death rate if measured. In other words, more sick patients get the most tests. If TroponinI was measured, it corresponds to 27.8 % mortality vs 13.1 %.

# AI Use Statement

AI tools were used substantially in the preparation. Tools used include Claude Code, ChatGPT, and GitHub Copilot. These tools were used to help summarize the data, generate code, and provide suggestions.

It is not a straightforward process. The main challenge is still instruction following. Claude was suprisingly good at catching extreme values and contextualizing them. Knowing that there might be conversion errors in height and realizing that a 733 pH is likely just 7.33 was good catch. 

What worked is when I provided as much context as possible to Claude. I could notice the contrast in performance when I asked ChatGPT about code suggestions without providing as much context. I ran all the code blocks here to make sure that they actually worked and outputted tables and results.

One example of how it still needs human oversight is when looking at the heatmap of when variables are measured. Claude suggested a normal heatmap that was not informative because it was comparing variables on the same scale given that it already knows that information.

AI was useful in helping with learning Git, which I was not familiar with.